# tm_final_34 — Final Production Pipeline

**Best model:** Twitter RoBERTa (fine-tuned, 3 epochs) + Random Forest  
**CV F1-macro:** 0.8641 | **CV Accuracy:** 0.8933

This notebook:
1. Trains the best model on **all** training data
2. Generates predictions for the test set → `results/pred_34.csv`
3. Exposes the trained model through the conversational agent

In [1]:
import os
import sys
import inspect
import tempfile

import numpy as np
import pandas as pd
import torch

from dotenv import load_dotenv
from sklearn.ensemble import RandomForestClassifier
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from transformers.utils import logging as hf_logging
from langchain_openai import AzureChatOpenAI

# Quiet Transformers output
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
hf_logging.set_verbosity_error()

# Simple path setup
PROJECT_ROOT = os.getcwd()
if not os.path.exists(os.path.join(PROJECT_ROOT, "data", "train.csv")):
    PROJECT_ROOT = os.path.abspath("..")

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from text_mining_utils import (
    LABEL_MAP,
    RANDOM_STATE,
    _HFTextDataset,
    preprocess_pipeline,
    detect_transformer_device,
    generate_cls_embeddings,
)
from agentic_tools import (
    build_agent_tools,
    create_specialist_executors,
    create_orchestrator_executor,
    create_orchestrator_memory_executor,
    print_trace,
)

# Azure config (professor-style) + key from .env
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))
AZURE_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_MODEL_NAME = "ChatGPT"
AZURE_ENDPOINT = "https://novaimsplayground.openai.azure.com/"
AZURE_API_VERSION = "2024-02-15-preview"

CHECKPOINT = "cardiffnlp/twitter-roberta-base"
FINE_TUNE_EPOCHS = 3
MAX_LENGTH = 96
BATCH_SIZE_EMBED = 16

device, device_name = detect_transformer_device()
torch_device = "cpu" if device == -1 else f"cuda:{device}"
print(f"Device: {device_name}")
print("Setup OK")

Device: cuda
Setup OK


## 1. Load Data

In [2]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train: {len(train_df):,} rows | Test: {len(test_df):,} rows")
print("Train columns:", train_df.columns.tolist())
print("Test  columns:", test_df.columns.tolist())
train_df.head(3)

Train: 9,543 rows | Test: 2,388 rows
Train columns: ['text', 'label']
Test  columns: ['id', 'text']


,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0


## 2. Preprocessing (Transformer Mode)

Same preprocessing used during experiments in `tm_tests_34`.

In [3]:
X_transformer_train = preprocess_pipeline(train_df["text"], mode="transformer")
X_transformer_test  = preprocess_pipeline(test_df["text"],  mode="transformer")
y_labels = train_df["label"]

print(f"Train samples: {len(X_transformer_train):,}")
print(f"Test  samples: {len(X_transformer_test):,}")
print(f"Label distribution:\n{y_labels.value_counts().sort_index().rename(LABEL_MAP)}")

Train samples: 9,543
Test  samples: 2,388
Label distribution:
label
Bearish    1442
Bullish    1923
Neutral    6178
Name: count, dtype: int64


## 3. Fine-tune Twitter RoBERTa on All Training Data

Replicating the exact setup from `tm_tests_34`:
- Checkpoint: `cardiffnlp/twitter-roberta-base`
- Epochs: 3, LR: 2e-5, weight decay: 0.01
- Batch size: 8 (GPU) / 4 (CPU)

In [4]:
y_arr = np.asarray(y_labels)

print("Tokenizing all training data...")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
train_enc = tokenizer(
    list(X_transformer_train),
    truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt",
)
train_ds = _HFTextDataset(train_enc, y_arr)

print("Loading Twitter RoBERTa for sequence classification...")
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=len(LABEL_MAP))
if torch.cuda.is_available():
    model.gradient_checkpointing_enable()

# Build TrainingArguments - no eval set (training on full data)
ta_params = inspect.signature(TrainingArguments.__init__).parameters
ft_out_dir = tempfile.mkdtemp(prefix="ft_roberta_final_")
training_kwargs = {
    "output_dir": ft_out_dir,
    "num_train_epochs": FINE_TUNE_EPOCHS,
    "per_device_train_batch_size": 8 if torch.cuda.is_available() else 4,
    "per_device_eval_batch_size": 16,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "save_strategy": "no",
    "logging_strategy": "no",
    "disable_tqdm": True,
    "report_to": [],
    "fp16": torch.cuda.is_available(),
}
# Disable evaluation for final full-data training
if "evaluation_strategy" in ta_params:
    training_kwargs["evaluation_strategy"] = "no"
else:
    training_kwargs["eval_strategy"] = "no"

training_args = TrainingArguments(**training_kwargs)

# Build Trainer
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "data_collator": DataCollatorWithPadding(tokenizer=tokenizer),
}
trainer_params = inspect.signature(Trainer.__init__).parameters
if "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print(f"\nFine-tuning for {FINE_TUNE_EPOCHS} epochs on {len(train_ds):,} samples...")
trainer.train()
print("\nFine-tuning complete!")

Tokenizing all training data...


Loading Twitter RoBERTa for sequence classification...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Fine-tuning for 3 epochs on 9,543 samples...
{'train_runtime': '180.6', 'train_samples_per_second': '158.5', 'train_steps_per_second': '19.81', 'train_loss': '0.3133', 'epoch': '3'}

Fine-tuning complete!


In [5]:
# ── Extract CLS embeddings from the fine-tuned encoder ──────────────────────
encoder = model.base_model

print("Extracting training embeddings from fine-tuned encoder...")
X_train_emb = generate_cls_embeddings(
    texts=list(X_transformer_train),
    tokenizer=tokenizer,
    model=encoder,
    batch_size=BATCH_SIZE_EMBED,
    max_length=MAX_LENGTH,
    desc="Train embeddings",
    device=torch_device,
)

# ── Fit Random Forest on all training embeddings ─────────────────────────────
print(f"\nFitting RandomForestClassifier (n_estimators=200, class_weight='balanced')...")
rf_classifier = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_classifier.fit(X_train_emb, y_arr)
print("RandomForest fitted on", X_train_emb.shape[0], "samples,", X_train_emb.shape[1], "features.")

Extracting training embeddings from fine-tuned encoder...


Train embeddings:   0%|          | 0/597 [00:00<?, ?it/s]


Fitting RandomForestClassifier (n_estimators=200, class_weight='balanced')...
RandomForest fitted on 9543 samples, 768 features.


## 4. Predict on Test Set → `results/pred_34.csv`

In [6]:
print("Extracting test embeddings from fine-tuned encoder...")
X_test_emb = generate_cls_embeddings(
    texts=list(X_transformer_test),
    tokenizer=tokenizer,
    model=encoder,
    batch_size=BATCH_SIZE_EMBED,
    max_length=MAX_LENGTH,
    desc="Test embeddings",
    device=torch_device,
)

print("Predicting test labels...")
y_test_pred = rf_classifier.predict(X_test_emb)

# Save predictions
os.makedirs(RESULTS_DIR, exist_ok=True)
pred_path = os.path.join(RESULTS_DIR, "pred_34.csv")
pred_df = pd.DataFrame({"id": test_df["id"], "label": y_test_pred})
pred_df.to_csv(pred_path, index=False)

print(f"\nSaved {len(pred_df):,} predictions -> {pred_path}")
print(f"Label distribution:\n{pred_df['label'].value_counts().sort_index().rename(LABEL_MAP)}")
pred_df.head(10)

Extracting test embeddings from fine-tuned encoder...


Test embeddings:   0%|          | 0/150 [00:00<?, ?it/s]

Predicting test labels...

Saved 2,388 predictions -> /home/lucas/Desktop/text_mining_project/results/pred_34.csv
Label distribution:
label
Bearish     363
Bullish     479
Neutral    1546
Name: count, dtype: int64


,id,label
0,0,1
1,1,2
2,2,2
3,3,1
4,4,2
5,5,2
6,6,0
7,7,0
8,8,2
9,9,2


In [7]:
# Save the fine-tuned base encoder so the agent can load it via pipeline
ft_encoder_dir = os.path.join(ft_out_dir, "encoder")
encoder.save_pretrained(ft_encoder_dir)
tokenizer.save_pretrained(ft_encoder_dir)
print(f"Fine-tuned encoder saved to: {ft_encoder_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned encoder saved to: /tmp/ft_roberta_final_xqwupkcr/encoder


# Agentic AI

We now expose a simplified agent workflow that uses the single best validated encoder-classifier model from the experiment results.

In [8]:
# Build the agent model registry from the trained model
# The best model (Twitter RoBERTa FT + Random Forest) is already trained above.
transformer_runs_agent = {
    "Twitter RoBERTa FT + Random Forest": {
        "checkpoint": ft_encoder_dir,   # fine-tuned encoder saved above
        "classifier": rf_classifier,
        "metrics": {
            "model": "Twitter RoBERTa FT + Random Forest",
            "cv_f1_macro_mean": 0.8641,
            "cv_f1_macro_std": 0.0050,
            "cv_accuracy_mean": 0.8933,
        },
    }
}

tools_bundle = build_agent_tools(
    transformer_runs=transformer_runs_agent,
    preprocess_pipeline_fn=preprocess_pipeline,
    label_map=LABEL_MAP,
    max_length=MAX_LENGTH,
    device=device,
)

MODEL_REGISTRY     = tools_bundle["model_registry"]
MODEL_NAME_LOOKUP  = tools_bundle["model_name_lookup"]
BEST_MODEL_NAME    = tools_bundle["best_model_name"]

inspect_tweet_profile    = tools_bundle["inspect_tweet_profile"]
get_validation_metrics   = tools_bundle["get_validation_metrics"]
classify_with_best_model = tools_bundle["classify_with_best_model"]

print(f"Best model selected for agent: {BEST_MODEL_NAME}")

Best model selected for agent: Twitter RoBERTa FT + Random Forest


In [9]:
# Tools are loaded from src/agentic_tools.py and ready to use
print("Loaded tools:")
print("- inspect_tweet_profile")
print("- get_validation_metrics")
print("- classify_with_best_model")
print(f"Best model selected for agent: {BEST_MODEL_NAME}")

Loaded tools:
- inspect_tweet_profile
- get_validation_metrics
- classify_with_best_model
Best model selected for agent: Twitter RoBERTa FT + Random Forest


### Building the Conversational Agent

The structure below is modular: define tools, create specialized agents, then expose a single conversational interface through an orchestrator.


In [10]:
# Load Azure credentials from .env key
os.environ["AZURE_OPENAI_KEY"] = AZURE_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = AZURE_ENDPOINT

# Initialise the LLM
llm = AzureChatOpenAI(
    temperature=0,
    model=AZURE_MODEL_NAME,
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    openai_api_type="azure",
    api_key=os.environ["AZURE_OPENAI_KEY"],
    api_version=AZURE_API_VERSION,
)
print("LLM ready")

LLM ready


In [11]:
# Create routing and classification specialist executors from src module
(
    routing_executor,
    classification_executor,
    routing_tools,
    classification_tools,
) = create_specialist_executors(llm, tools_bundle)

print("Routing tools:", [tool.name for tool in routing_tools])
print("Classification tools:", [tool.name for tool in classification_tools])

Routing tools: ['inspect_tweet_profile', 'get_validation_metrics']
Classification tools: ['classify_with_best_model']


In [12]:
# Create orchestrator executor from src module
orchestrator_executor, orchestrator_tools = create_orchestrator_executor(
    llm=llm,
    routing_executor=routing_executor,
    classification_executor=classification_executor,
)

print("Orchestrator tools:", [tool.name for tool in orchestrator_tools])
print("Agent ready.")

Orchestrator tools: ['routing_expert', 'classification_expert']
Agent ready.


In [13]:
# Run the orchestrator - watch the routing - classification coordination in the output

tweet = "$TSLA is finally breaking out after earnings! Loading more shares now  #bullish"

agentic_result = orchestrator_executor.invoke(
    {
        "input": (
            f"Classify this investor tweet: '{tweet}'. "
            "First choose the most appropriate classifier strategy, then give me the final label, "
            "confidence, and a short justification."
        )
    }
)

print("\n" + "-" * 62)
print("Answer:", agentic_result["output"])
print("-" * 62)


AuthenticationError: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}

In [ ]:
print_trace(agentic_result)


### Automated Evaluation Through the Same Conversational Interface

The same agent can answer evaluation prompts as well, using the validation metrics tool instead of hardcoded notebook logic.


In [ ]:
evaluation_result = orchestrator_executor.invoke(
    {
        "input": (
            "Compare the available classifiers on the validation set and recommend which one we should deploy "
            "for noisy stock-market tweets with hashtags and cashtags."
        )
    }
)

print("\n" + "-" * 62)
print("Answer:", evaluation_result["output"])
print("-" * 62)


In [ ]:
print_trace(evaluation_result)


### Optional Memory for Follow-up Questions

We can also add short-term conversational memory so the user can ask follow-up questions about the previously recommended strategy.


In [ ]:
orchestrator_executor_mem = create_orchestrator_memory_executor(
    llm=llm,
    orchestrator_tools=orchestrator_tools,
)

print("Memory agent ready.")

In [ ]:
q1 = "For tweets full of cashtags and hashtags, which model strategy should we prefer- Remember your recommendation."
print(f"User  : {q1}")
ans1 = orchestrator_executor_mem.invoke({"input": q1})
print(f"Agent : {ans1['output']}")
print_trace(ans1)

print()

q2 = "Now classify this tweet using that recommendation: '$NVDA looks unstoppable right now #AI #stocks'"
print(f"User  : {q2}")
ans2 = orchestrator_executor_mem.invoke({"input": q2})
print(f"Agent : {ans2['output']}")
print_trace(ans2)
